# 3.1 Churn Prediction Project

Predict customer churn for a telecom company.

### Problem Statement: The Problem of Churn

Churn is the rate at which telco customers leave for another provider, calculated on a monthly basis. We will compute a score from 0 to 1 as the probablity that a customer is about to churn, before they actually do.

### Classification: Binary

We will predict this score with machine learning. For the $i-th$ customer, we want our model $g$ to approximate the target $y$. The task is binary classification: We look at one customer and predict one of two possible outcomes.

$$
g(x_{i}) \approx y_{i}
$$

$x$ is the feature vector, which is everything that we know about the customer. Being binary, $y$ belongs to `{0, 1}`. 

- 1 is the positive class: The customer churned.
- 0 is the negative class: The customer stayed.

We learn $g$ from historical data. We take the customers of the previous month as the source of data, because we already know what happened to them. 

For each of them we know the demographics, what services they use, what kind of contract they have and how much they pay, and we know whether they stayed or left. 

The model finds the patterns in this data, and then we apply it to the current customers to score them

### The Dataset

Telco Customer Churn
https://www.kaggle.com/datasets/blastchar/telco-customer-churn

## 3.2 Data Preparation


In [2]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
# %matplotlib inline

### Download and Read In the Data

In [ ]:
data = 'https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-03-churn-prediction/WA_Fn-UseC_-Telco-Customer-Churn.csv'

!wget $data -O data-week-3.csv

In [3]:
df = pd.read_csv('data-week-3.csv')
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


### Data Preparation: Make Everything Uniform

In [4]:
df.columns = df.columns.str.lower().str.replace(' ', '_')

categorical_columns = list(df.dtypes[df.dtypes == 'str'].index)

for c in categorical_columns:
    df[c] = df[c].str.lower().str.replace(' ', '_')

### Fixing the totalcharges Column

In [5]:
df.dtypes

customerid              str
gender                  str
seniorcitizen         int64
partner                 str
dependents              str
tenure                int64
phoneservice            str
multiplelines           str
internetservice         str
onlinesecurity          str
onlinebackup            str
deviceprotection        str
techsupport             str
streamingtv             str
streamingmovies         str
contract                str
paperlessbilling        str
paymentmethod           str
monthlycharges      float64
totalcharges            str
churn                   str
dtype: object

- `seniorcitizen` is `int64`
- `totalcharges` is `str`

When we try to convert it to numbers, it fails.

In [5]:
pd.to_numeric(df.totalcharges)

ValueError: Unable to parse string "_" at position 488

In [5]:
# Unparseable strings will be replaced with NaN

tc = pd.to_numeric(df.totalcharges, errors='coerce')

In [6]:
tc.isnull().sum()

np.int64(11)

In [7]:
df[tc.isnull()][['customerid', 'totalcharges']]

,customerid,totalcharges
488,4472-lvygi,_
753,3115-czmzd,_
936,5709-lvoeq,_
1082,4367-nuyao,_
1340,1371-dwpaz,_
3331,7644-omvmy,_
3826,3213-vvolg,_
4380,2520-sgtta,_
5218,2923-arzlg,_
6670,4075-wkniu,_


These are customers who just joined - their tenure is small and they haven't been billed yet, so their total charges are not available. For them we simply fill the missing values with zero.

In [8]:
df.totalcharges = pd.to_numeric(df.totalcharges, errors='coerce')
df.totalcharges = df.totalcharges.fillna(0)

### Encoding the Target Value

The target `churn` still contains strings.

In [10]:
df.churn.head()

0     no
1     no
2    yes
3     no
4    yes
Name: churn, dtype: str

Below, if df.churn field is `yes`, this yields a boolean value, which is then cast to integers

In [9]:
df.churn = (df.churn == 'yes').astype(int)

In [10]:
df.churn.head()

0    0
1    0
2    1
3    0
4    1
Name: churn, dtype: int64

## 2.3 Validation

We continue by setting up the validation framework

In [11]:
from sklearn.model_selection import train_test_split

In [12]:
# The first 20% is set aside as the test set
# random_state=1 fixes the seed of the random shuffling

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)

# The remaining 80% -- referred to as the full train set -- will be split into train and validation
# Note that 20% of 80% is 25%, or as I would say, 20% of the original is 25% of the 80%

df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

# Please let me know how things went

print(f"df_train: {len(df_train)}, df_val: {len(df_val)}, df_test: {len(df_test)}")

df_train: 4225, df_val: 1409, df_test: 1409


In [13]:
len(df)

7043

### Reset the Index

`drop=True` discards the old index instead of adding back as a column.

In [14]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

To prevent accidentally using the targete as a feature when we train the model, we delete it from the dataframes that we created.

In [15]:
del df_train['churn']
del df_val['churn']
del df_test['churn']

## 3.4 EDA Analysts on Safari

Generally we become familar with the data. Specifically we: 
1. Look for missing values
2. Make a determination on the distribution of the target variable
3. Peruse the numerical and categorial columns

In [16]:
df_full_train.head()

,customerid,gender,seniorcitizen,partner,dependents,tenure,phoneservice,multiplelines,internetservice,onlinesecurity,...,deviceprotection,techsupport,streamingtv,streamingmovies,contract,paperlessbilling,paymentmethod,monthlycharges,totalcharges,churn
1814,5442-pptjy,male,0,yes,yes,12,yes,no,no,no_internet_service,...,no_internet_service,no_internet_service,no_internet_service,no_internet_service,two_year,no,mailed_check,19.70,258.35,0
5946,6261-rcvns,female,0,no,no,42,yes,no,dsl,yes,...,yes,yes,no,yes,one_year,no,credit_card_(automatic),73.90,3160.55,1
3881,2176-osjuv,male,0,yes,no,71,yes,yes,dsl,yes,...,no,yes,no,no,two_year,no,bank_transfer_(automatic),65.15,4681.75,0
2389,6161-erdgd,male,0,yes,yes,71,yes,yes,dsl,yes,...,yes,yes,yes,yes,one_year,no,electronic_check,85.45,6300.85,0
3676,2364-ufrom,male,0,no,no,30,yes,no,dsl,yes,...,no,yes,yes,no,one_year,no,electronic_check,70.40,2044.75,0


In [17]:
df_full_train = df_full_train.reset_index(drop=True)

### Check for Missing Values

In [18]:
df_full_train.isnull().sum()

customerid          0
gender              0
seniorcitizen       0
partner             0
dependents          0
tenure              0
phoneservice        0
multiplelines       0
internetservice     0
onlinesecurity      0
onlinebackup        0
deviceprotection    0
techsupport         0
streamingtv         0
streamingmovies     0
contract            0
paperlessbilling    0
paymentmethod       0
monthlycharges      0
totalcharges        0
churn               0
dtype: int64

The missing values that existed were actually taken care of during data preparation in the `totalcharges` column.

### Looking at the Target: The Churn Rate

In [19]:
df_full_train.churn.value_counts(normalize=True)

churn
0    0.730032
1    0.269968
Name: proportion, dtype: float64

These numbers are in percentages, so about 73% stayed and 27% churned (left).

As below, the churn rate can also be calculated using the mean of these binary values.

In [20]:
df_full_train.churn.mean()

np.float64(0.26996805111821087)

### Numerical and Categorical Variables

We will operate under the rule-of-thumb that string columns are categorical.

Eliminating `seniorcitizen` and `churn`, leaves three (3) numerical columns.

The remaining 16 are therefore categorical columns. `customerid` is excluded because it is an identifier, and is thus useless as a feature.

Lastly for this unit we will check how many unique values each categorical variable has.

In [21]:
numerical = ['tenure', 'monthlycharges', 'totalcharges']

categorical = [
    'gender',
    'seniorcitizen',
    'partner',
    'dependents',
    'phoneservice',
    'multiplelines',
    'internetservice',
    'onlinesecurity',
    'onlinebackup',
    'deviceprotection',
    'techsupport',
    'streamingtv',
    'streamingmovies',
    'contract',
    'paperlessbilling',
    'paymentmethod',
]

df_full_train[categorical].nunique()

gender              2
seniorcitizen       2
partner             2
dependents          2
phoneservice        2
multiplelines       3
internetservice     3
onlinesecurity      3
onlinebackup        3
deviceprotection    3
techsupport         3
streamingtv         3
streamingmovies     3
contract            3
paperlessbilling    2
paymentmethod       4
dtype: int64

The number of categories matters later: it determines how many columns we get after one-hot encoding.

## 3.5 Feature Importance: Churn Rate and Risk Ratio

Feature importance analysis is a part of EDA. We want to identify which features affect our target variable.

We do it for categorical variables by comparing the churn rate of each group with the global churn rate, first as a difference and then as a ratio.

In [22]:
# Global Churn Rate

global_churn = df_full_train.churn.mean()
global_churn

np.float64(0.26996805111821087)

In [23]:
# Churn Rate Per Group

churn_female = df_full_train[df_full_train.gender == 'female'].churn.mean()
churn_male = df_full_train[df_full_train.gender == 'male'].churn.mean()

print(f'Churn Female: {churn_female}, Churn Male: {churn_male}')

Churn Female: 0.27682403433476394, Churn Male: 0.2632135306553911


In [27]:
print(f'global_churn - churn_female: {global_churn - churn_female}, global_churn - churn_male: {global_churn - churn_male
}')

global_churn - churn_female: -0.006855983216553063, global_churn - churn_male: 0.006754520462819769


Both differences are close to zero, so gender is probably not an important feature. Knowing whether a customer is a man or a woman tells us almost nothing about their likely churn.

Next we look at whether or not a customer has a partner.

In [29]:
df_full_train.partner.value_counts()

partner
no     2932
yes    2702
Name: count, dtype: int64

In [24]:
churn_partner = df_full_train[df_full_train.partner == 'yes'].churn.mean()
churn_no_partner = df_full_train[df_full_train.partner == 'no'].churn.mean()

print(f'Churn Partner: {churn_partner}, Churn No Partner: {churn_no_partner}')

Churn Partner: 0.20503330866025166, Churn No Partner: 0.3298090040927694


In [31]:
# Compare these churn rates 20.5% and 33% to the global rate

print(f'Global v. Partner: {global_churn - churn_partner}, Global v. No Partner: {global_churn - churn_no_partner
}')

Global v. Partner: 0.06493474245795922, Global v. No Partner: -0.05984095297455855


### Risk Ratio

The differences shown above are expressed in absolute terms. It is also to express this in relative terms, which is a ratio betwen the group churn rate and the global churn rate.

Different ways of expressing **Feature Importance**:
(1) Difference

    Global minus Group

        < 0, more likely to churn
        > 0, less likely to churn
    
(2) Risk Ratio

    Risk is Group divided by Global

        > 1, more likely to churn
        < 1, less likely to churn

In [32]:
churn_no_partner / global_churn

np.float64(1.2216593879412643)

In [33]:
churn_partner / global_churn

np.float64(0.7594724924338315)

Customers without a partner have a risk of 1.22, and are 22% more likely to churn than the average customer.

Customers with a parthner have a risk of 0.76, and are 24% less likely to churn than the average customer.

If this were expressed as an SQL query:

In [ ]:
SELECT
    gender,
    AVG(churn),
    AVG(churn) - global_churn AS diff,
    AVG(churn) / global_churn AS risk
FROM
    data
GROUP BY
    gender;

In pandas, a SQL `GROUP BY` is done with aggregation.

In [25]:
from IPython.display import display

for c in categorical:
    print(c)
    df_group = df_full_train.groupby(c).churn.agg(['mean', 'count'])
    df_group['diff'] = df_group['mean'] - global_churn
    df_group['risk'] = df_group['mean'] / global_churn
    display(df_group)
    print()
    print()

gender


,mean,count,diff,risk
gender,,,,
female,0.276824,2796,0.006856,1.025396
male,0.263214,2838,-0.006755,0.974980




seniorcitizen


,mean,count,diff,risk
seniorcitizen,,,,
0,0.242270,4722,-0.027698,0.897403
1,0.413377,912,0.143409,1.531208




partner


...

,mean,count,diff,risk
contract,,,,
month-to-month,0.431701,3104,0.161733,1.599082
one_year,0.120573,1186,-0.149395,0.446621
two_year,0.028274,1344,-0.241694,0.104730




paperlessbilling


,mean,count,diff,risk
paperlessbilling,,,,
no,0.172071,2313,-0.097897,0.637375
yes,0.338151,3321,0.068183,1.252560




paymentmethod


,mean,count,diff,risk
paymentmethod,,,,
bank_transfer_(automatic),0.168171,1219,-0.101797,0.622928
credit_card_(automatic),0.164339,1217,-0.105630,0.608733
electronic_check,0.455890,1893,0.185922,1.688682
mailed_check,0.193870,1305,-0.076098,0.718121


## 3.6 Feature Importance: Mutual Information

**Mutual Information** allows us to compare whole variables with each other. It is a single number per variable that measures how important it is. It is a concept from information theory that tells us how much we can learn about one variable if we know the value of another variable.

If knowing a feature always gives us useful information about churn, the mutual information between them is high and the feature is important. If the feature tells us nothing, the mutual information is 0.

The order of arguments does not matter because mutual information is symmetric.

In [26]:
from sklearn.metrics import mutual_info_score

In [27]:
# For contract

mutual_info_score(df_full_train.churn, df_full_train.contract)

0.0983203874041556

In [28]:
# For gender

mutual_info_score(df_full_train.gender, df_full_train.churn)

0.0001174846211139946

In [29]:
# For partner

mutual_info_score(df_full_train.partner, df_full_train.churn)

0.009967689095399745

### Apply Mutual Information to All Categorical Variables

In [30]:
def mutul_info_churn_score(series):
    return mutual_info_score(series, df_full_train.churn)

Now apply it to every categorical column and sort the result from most to least imporant.

In [31]:
mi = df_full_train[categorical].apply(mutul_info_churn_score)
mi.sort_values(ascending=False)

contract            0.098320
onlinesecurity      0.063085
techsupport         0.061032
internetservice     0.055868
onlinebackup        0.046923
deviceprotection    0.043453
paymentmethod       0.043210
streamingtv         0.031853
streamingmovies     0.031581
paperlessbilling    0.017589
dependents          0.012346
partner             0.009968
seniorcitizen       0.009410
multiplelines       0.000857
phoneservice        0.000229
gender              0.000117
dtype: float64

## 3.7 Feature Importance: Correlation

Mutual information measures the importance of categorical variables. For numerical variables we use the correlation coefficient instead. It tells us how strongly each numerical feature is related to churn.

The *Pearson correlation coefficient** is usually denoted $r$. It measures the degree of dependency between two variables. It is always a number between -1 and 1.

- A value close to 1 means a strong positive correlation
- A value close to -1 means a strong negative correlation
- A value close to 0 means there is no dependency; a change in one value tells us nothing about the other.

The sign gives the direction of the relationship. The absolute value give its strength. Magnitude can be interpreted:

- **Low** when the absolute value of $r$ is between 0 and 0.2
- **Medium** when the absolute value of $r$ is between 0.2 and 0.5
- **Strong** when the absolute value of $r$ is between 0.5 and 1.0

If the correlation between a numerical feature and churn is positive, customers with higher values of that feature churn more; if it is negative, they churn less. 

The larger the absolute value, the more important the feature.

### Correlation of Numerical Features with Churn

In [32]:
df_full_train[numerical].corrwith(df_full_train.churn)

tenure           -0.351885
monthlycharges    0.196805
totalcharges     -0.196353
dtype: float64

- Tenure has a medium negative correlation. The longer customers stay with the company, the less likely they are to churn.
- Monthly charges have a low-to-medium positive correlation: the more customers pay per month, the more likely they are to churn.
- Total charges have a low-to-medium negative correlation - not surprising, since total charges accumulate with tenure.

To rank features by importance we don't care about the direction, only about the strength, so we take the absolute values.

In [33]:
df_full_train[numerical].corrwith(df_full_train.churn).abs()

tenure            0.351885
monthlycharges    0.196805
totalcharges      0.196353
dtype: float64

### Checking the Relationship with Groups

Check the correlation by splitting customers into groups and computing the churn rate in each.

Tenure is measured in months and goes up to 72.

In [34]:
df_full_train.tenure.max()

np.int64(72)

Split it into three groups:
- Ttwo months or less
- Up to a year
- Longer than a year

In [35]:
df_full_train[df_full_train.tenure <= 2].churn.mean()

np.float64(0.5953420669577875)

In [36]:
df_full_train[(df_full_train.tenure > 2) & (df_full_train.tenure <= 12)].churn.mean()

np.float64(0.3994413407821229)

In [37]:
df_full_train[df_full_train.tenure > 12].churn.mean()

np.float64(0.17634908339788277)

The same check for monthly charges.

In [38]:
df_full_train[df_full_train.monthlycharges <= 20].churn.mean()

np.float64(0.08795411089866156)

In [39]:
df_full_train[(df_full_train.monthlycharges > 20) & (df_full_train.monthlycharges <= 50)].churn.mean()

np.float64(0.18340943683409436)

In [40]:
df_full_train[df_full_train.monthlycharges > 50].churn.mean()

np.float64(0.32499341585462205)

## 3.8 OHE - One-Hot Encoding

Here we encode categorical features as numbers so that they can be used by a machine learning model. 

Scikit-Learn gives us a convenient way to do it with `DictVectorizer`.